# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR<sup>2</sup> dataset on mass spectrometry of extracellular vesicles using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL and adheres to the FAIR principles for machine learning data packaging.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Let's load metadata and inspect the high-level description of the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Initialize the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

We will now review all available record sets, their `@id`s, and the fields contained in each. This helps us understand the structure of the dataset and what data is available for further processing.

#### List all record sets

In [ ]:
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"- RecordSet: name='{rs.name}', @id='{rs.id_}'")

#### List fields for each record set
Here, we display the fields (and their `@id`s) available in each record set. This is useful for referencing specific fields by `@id` in later steps.

In [ ]:
for rs in record_sets:
    print(f"\nRecordSet: {rs.name} (@id={rs.id_})")
    print("Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id={field.id_}, type={field.data_type})")

## 3. Data Extraction

We will load all available record sets into separate pandas DataFrames. All referencing uses record set and field `@id` values for accuracy and reproducibility.

In [ ]:
# Get the @id for each record set
record_set_ids = [rs.id_ for rs in dataset.record_sets]
dataframes = {}

# Load data for each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet @id='{record_set_id}' with {len(df)} rows.")

Let's display the column names of the main record set. **Choose one record set (e.g., the first) to use in forthcoming analysis.**

In [ ]:
# Example: Use the first record set for demonstration
main_record_set_id = record_set_ids[0]
print(f"Columns for main record set (@id='{main_record_set_id}'):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We will select a numeric field by its `@id` (using the printed field lists above), filter, normalize, and group the data, all referencing via `@id`.

In [ ]:
# Select a numeric field's @id from the main record set for demonstration (replace with real @id as appropriate)

# EXAMPLE: Let's assume 'cr:coverage_percentage' is a numeric field within the main record set
numeric_field_id = 'cr:coverage_percentage'  # Replace with actual numeric field @id from field list if needed

df = dataframes[main_record_set_id]

# Convert to float if necessary
if numeric_field_id in df.columns:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Filter: coverage > 10 (example threshold)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optionally, group by a categorical field (replace with an actual @id from your overview, e.g. 'cr:protein_family')
    group_field_id = 'cr:protein_family'  # Replace with a valid group field @id, if exists
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field 
(coverage percentage) for the filtered proteins.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=25, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} (Filtered > {threshold})")
    plt.show()

## 6. Conclusion

Using the `mlcroissant` library, we've loaded a mass spectrometry dataset in standardized Croissant format, explored its structure, filtered high-coverage proteins, normalized numeric data, and visualized distributions. Refer to each entity by their Croissant `@id` for reproducibility. You can now extend this notebook for further processing, custom visualizations, or in-depth comparative analysis.